# Training model


## Verification of dependencies and their versions

In [1]:
import time
from datetime import datetime

import kubeflow
print(f"Kubeflow version: {kubeflow.__version__ if hasattr(kubeflow, '__version__') else 'N/A'}")
print("All imports successful!")

Kubeflow version: 0.3.0
All imports successful!


## PyTorch DDP with Kubeflow TrainJob

You can use `TrainerClient()` from the Kubeflow SDK to communicate with Kubeflow Trainer APIs and scale your training function across multiple PyTorch training nodes. `TrainerClient()` verifies that you have required access to the Kubernetes cluster. Kubeflow Trainer creates a `TrainJob` resource and automatically sets the appropriate environment variables to set up PyTorch in distributed environment.



In [2]:

from kubeflow.trainer import CustomTrainer, TrainerClient
config = kubeflow.trainer.KubernetesBackendConfig()

client = TrainerClient()
trainer = TrainerClient()
# trainer = TrainerClient(backend_config=config) #TOTEST

## List the Training Runtimes

You can get the list of available Training Runtimes to start your TrainJob. Additionally, it might show available accelerator type and number of available resources.

In [3]:
for runtime in trainer.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

Runtime(name='deepspeed-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='deepspeed', image='ghcr.io/kubeflow/trainer/deepspeed-runtime:v2.1.0', num_nodes=1, device='Unknown', device_count='1'), pretrained_model=None)
Runtime(name='mlx-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='mlx', image='ghcr.io/kubeflow/trainer/mlx-runtime:v2.1.0', num_nodes=1, device='Unknown', device_count='1'), pretrained_model=None)
Runtime(name='retfound-image-generation', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='ghcr.io/andesterson/yukun-image-generation-runner:v0.0.8', num_nodes=1, device='gpu', device_count='1'), pretrained_model=None)
Runtime(name='retfound-pretraining', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='ghcr.io/andesterson/dinov3-kubeflow-train

## Run the Distributed TrainJob

Kubeflow TrainJob will train the above model on PyTorch nodes defined by `NUM_NODES` and each node with `RESOURCES_PER_NODE`.

In [4]:
from kubeflow.trainer import TrainerClient, CustomTrainer, options
from kubeflow.trainer.options import TrainerCommand
from kubeflow.trainer.types.types import CustomTrainerContainer


# ------------------------------------------------
# Configuration of resources
# ------------------------------------------------

## Set how many PyTorch nodes you want to use for distributed training.
NUM_NODES = 1

# Set the resources for each PyTorch node.
RESOURCES_PER_NODE = {
    "cpu": "5",           # CPUs per node
    "memory": "2Gi",     # Memory in GiB per node
    "nvidia.com/gpu": 1,  # GPUs per node (the number will depend on the available resources)
}

# Set github container registry
GITHUB_CONTAINER_REGISTRY = "ghcr.io/xfetus/fetal-ultrasound-edm2/fetal-ultrasound-edm2-distributed-learning:v0.0.2"

# Set up arguments of your function
FUNC_ARGS={
    "checkpoint_path": "/scratch-volume/YOUR_CHECKPOINT_PATH", # pragma: allowlist secret
    "save_interval": 10,
    "resume": False,
    "epochs": 50,
    "batch_size": 128,
}

# ------------------------------------------------
# Seeting up to mount checkpoint volume
# ------------------------------------------------
volume_name = "scratch-volume"
pvc_name = "scratch-volume"
mount_scrath_path = "/scratch-volume"

pod_volumes = [
    {
        "name": volume_name,
        "persistentVolumeClaim": {
            "claimName": pvc_name
        }
    }
]
volume_mounts = [
    {
        "name": volume_name,
        "mountPath": mount_scrath_path
    }
]
container_override = options.ContainerOverride(
    name="node",
    volume_mounts=volume_mounts
)
pod_spec_override = options.PodSpecOverride(
    volumes=pod_volumes,
    containers=[container_override]
)
pod_template_override = options.PodTemplateOverride(
    target_jobs=["node"],
    spec=pod_spec_override
)
pod_template_overrides = options.PodTemplateOverrides(
    pod_template_override
)


ENV_VARS = {
    "MASTER_ADDR": "localhost",
    "MASTER_PORT": "12355",
    "WORLD_SIZE": "1",
    "RANK": "0",
    "LOCAL_RANK": "0"
}

command = TrainerCommand(
    command=[
        "torchrun",
        "--standalone",
        "--nproc_per_node=1",
        "train_edm2.py",
        "--outdir", "~/scratch-volume/FETAL_PLANES_ZENODO/OUTPUT_DIRECTORY",
        "--data", "~/scratch-volume/FETAL_PLANES_ZENODO",
        "--batch", "8",
        "--preset", "edm2-img512-s",
        "--batch-gpu", "8",
    ]
)


In [5]:
job_id = trainer.train(
    runtime=torch_runtime,
    trainer=CustomTrainerContainer(
        image=GITHUB_CONTAINER_REGISTRY,
        num_nodes=NUM_NODES,
        resources_per_node=RESOURCES_PER_NODE,
        env=ENV_VARS        
    ),
    options=[command, pod_template_overrides],
)


In [6]:
#Check job status directly
job = trainer.get_job(job_id)
print(f"\nJob ID: {job_id}")
print(f"Job Status: {job.status}")
print(f"Creation Time: {job.creation_timestamp}")
print(f"\nJob details: {job}")


Job ID: te2efc265b5c
Job Status: Created
Creation Time: 2026-04-01 14:23:44+00:00

Job details: TrainJob(name='te2efc265b5c', runtime=Runtime(name='torch-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='pytorch/pytorch:2.7.1-cuda12.8-cudnn9-runtime', num_nodes=1, device='Unknown', device_count='Unknown'), pretrained_model=None), steps=[], num_nodes=1, creation_timestamp=datetime.datetime(2026, 4, 1, 14, 23, 44, tzinfo=TzInfo(0)), status='Created')


In [7]:
from datetime import datetime
import time
print("Waiting for job logs...")
wait_count = 0

while True:
    initial_logs = list(trainer.get_job_logs(job_id, follow=True))
    if initial_logs:
        print(f"Logs received after {wait_count} seconds:")
        for log in initial_logs:
            print(f"  {log}")
        break
    
    wait_count += 1
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Waiting... ({wait_count}s)")
    time.sleep(1)


Waiting for job logs...
[14:23:45] Waiting... (1s)
[14:23:46] Waiting... (2s)
[14:23:47] Waiting... (3s)
[14:23:48] Waiting... (4s)
Logs received after 4 seconds:
  Traceback (most recent call last):
    File "/workspace/train_edm2.py", line 18, in <module>
      import training.training_loop as training_loop
    File "/workspace/training/__init__.py", line 8, in <module>
      from .dataset import *  # noqa: F403
      ^^^^^^^^^^^^^^^^^^^^^^
    File "/workspace/training/dataset.py", line 17, in <module>
      import pandas as pd
  ModuleNotFoundError: No module named 'pandas'
  E0401 14:23:52.136000 1 site-packages/torch/distributed/elastic/multiprocessing/api.py:874] failed (exitcode: 1) local_rank: 0 (pid: 57) of binary: /opt/conda/bin/python3.11
  Traceback (most recent call last):
    File "/opt/conda/bin/torchrun", line 8, in <module>
      sys.exit(main())
               ^^^^^^
    File "/opt/conda/lib/python3.11/site-packages/torch/distributed/elastic/multiprocessing/errors/__

# Delete the TrainJob
When TrainJob is finished, you can delete the resource.

In [8]:
client.delete_job(job_id)